In [1]:
#Let’s import the libraries we’ll be using.

import numpy as np
import scipy
import pandas
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set_style('white')
sns.set_style('ticks')
sns.set_context('notebook')
import h5py
import allel
import zarr
import matplotlib.patches as patches  # <-- This is the missing import
import pickle
import pandas as pd

In [2]:
chrom_regions='/home/usriniva/Desktop/masters/plasmodium/find_regions_recomb/regions.txt'
chrom_regions= pd.read_csv(chrom_regions, sep='\t',names=['chrom', 'start', 'end', 'region_type'])
chrom_regions.tail()

,chrom,start,end,region_type
105,Pf3D7_14_v3,35775,1071523,Core
106,Pf3D7_14_v3,1071524,1075089,Centromere
107,Pf3D7_14_v3,1075090,3255710,Core
108,Pf3D7_14_v3,3255711,3291511,SubtelomericHypervariable
109,Pf3D7_14_v3,3291512,3291936,SubtelomericRepeat


In [3]:
# Calculate total length of each chromosome
chrom_lengths = chrom_regions.groupby('chrom')['end'].max()

# Convert chrom_lengths (Series) to DataFrame
chrom_lengths = pd.DataFrame(chrom_lengths)

# Display total chromosome lengths
print(chrom_lengths)

                 end
chrom               
Pf3D7_01_v3   640851
Pf3D7_02_v3   947102
Pf3D7_03_v3  1067971
Pf3D7_04_v3  1200490
Pf3D7_05_v3  1343577
Pf3D7_06_v3  1418242
Pf3D7_07_v3  1445207
Pf3D7_08_v3  1472805
Pf3D7_09_v3  1541735
Pf3D7_10_v3  1687656
Pf3D7_11_v3  2038340
Pf3D7_12_v3  2271494
Pf3D7_13_v3  2925236
Pf3D7_14_v3  3291936


In [3]:
vcf_file_path_full = '/home/usriniva/Desktop/masters/NASmountpoint/Plasmodium_us/Inputs/VCFs/vcf_gz/vcg.gz.r3/vcfs_w_newmissfilt_newpops/Nofilt/Pf3D7_Nofilt_qcpass_White_combined.vcf.gz'
zarr_full = '/home/usriniva/Desktop/masters/zarrs/AllChr_qcpass_Nofilt_White_combined.zarr'
allel.vcf_to_zarr(vcf_file_path_full, zarr_full, group='All_chr', fields='*', transformers=allel.ANNTransformer(),overwrite=True)



In [4]:
zarr_full = '/home/usriniva/Desktop/masters/zarrs/AllChr_qcpass_Nofilt_White_combined.zarr'
callset_full = zarr.open_group(zarr_full, mode='r')



In [5]:
# Load the right table
samples_p = '/home/usriniva/Desktop/masters/plasmodium/VCFs/2_Samplemd_files/Final_Dataset_bydapcclusters_remoutliers/Allchr_Whitefullfilt_Newdapc_bycluster_md.txt'
samples_right = pd.read_csv(samples_p, sep='\t')
print('No of Individuals (right table):', len(samples_right))

# Add 'subpops' information based on 'assgn_clus'
samples_right['subpops'] = 'NA'  # Default value for unmatched samples
samples_right.loc[samples_right['assgn_clus'] == 1, 'subpops'] = 'MaeSot_Thailand'
samples_right.loc[samples_right['assgn_clus'] == 6, 'subpops'] = 'PurPai_Cambodia'
samples_right.loc[samples_right['assgn_clus'] == 4, 'subpops'] = 'BinhPhuoc_Long_Vietnam'

# Load the left table
samples_o_p = '/home/usriniva/Desktop/masters/plasmodium/VCFs/2_Samplemd_files/Total_dataset_TRACwhite2011/White_1052_ctv_471inds_depthfilt_samplemd.txt'
samples_left = pd.read_csv(samples_o_p, sep='\t')
print('No of Individuals (left table):', len(samples_left))

# Merge the tables using a left join: match 'Sample' from the left table with 'Individuals' from the right
merged = samples_left.merge(samples_right[['Individual', 'subpops']], how='left', left_on='Sample', right_on='Individual')

# Verify the data type of 'assgn_clus'
print(f"Data type of 'subpops': {merged['subpops'].dtype}")

# Check if 'assgn_clus' contains only the values 1, 4, 6
print(merged['subpops'].value_counts())
# Define subpopulations based on 'subpops' column in the merged DataFrame
subpops = {
    'all': list(range(len(merged))),  # All individuals in the merged dataset
    'MaeSot_Thailand': merged[merged['subpops'] == "MaeSot_Thailand"].index.tolist(),
    'PurPai_Cambodia': merged[merged['subpops'] == "PurPai_Cambodia"].index.tolist(),
    'BinhPhuoc_Long_Vietnam': merged[merged['subpops'] == "BinhPhuoc_Long_Vietnam"].index.tolist()
}


No of Individuals (right table): 270
No of Individuals (left table): 448
Data type of 'subpops': str
subpops
BinhPhuoc_Long_Vietnam    156
MaeSot_Thailand            89
PurPai_Cambodia            25
Name: count, dtype: int64


In [6]:
gt_zarr = callset_full['All_chr/calldata/GT']
gt_zarr.info
gt = allel.HaplotypeArray(callset_full['All_chr/calldata/GT'][:,:,0])
gt
genotypes_full = allel.HaplotypeChunkedArray(gt)
print(len(genotypes_full))

variants_full = allel.VariantChunkedTable(callset_full['All_chr/variants'], 
                                     names=['POS', 'CHROM', 'numalt', 'REF', 'ALT', 'DP', 'MQ', 'QD','ANN_Annotation'])
gt_full = allel.HaplotypeArray(callset_full['All_chr/calldata/GT'][:,:,0])
print(gt.shape)
genotypes_full = allel.HaplotypeChunkedArray(gt_full)


482115
(482115, 471)


In [10]:
# Create an ArrayWrapper object from the variants.CHROM array
chrom_uniq = np.unique(variants_full['CHROM'])
chrom_uniq

array(['Pf3D7_01_v3', 'Pf3D7_02_v3', 'Pf3D7_03_v3', 'Pf3D7_04_v3',
       'Pf3D7_05_v3', 'Pf3D7_06_v3', 'Pf3D7_07_v3', 'Pf3D7_08_v3',
       'Pf3D7_09_v3', 'Pf3D7_10_v3', 'Pf3D7_11_v3', 'Pf3D7_12_v3',
       'Pf3D7_13_v3', 'Pf3D7_14_v3'], dtype=object)

In [11]:
variants_full['CHROM'][:]

array(['Pf3D7_01_v3', 'Pf3D7_01_v3', 'Pf3D7_01_v3', ..., 'Pf3D7_14_v3',
       'Pf3D7_14_v3', 'Pf3D7_14_v3'], shape=(482115,), dtype=object)

In [9]:
regions='/home/usriniva/Desktop/masters/plasmodium/find_regions_recomb/regions.txt'

regions= pd.read_csv(regions, sep='\t', names=['region_chrom', 'region_start', 'region_stop', 'region_type'])

regions_non_accessible = regions[
    (regions['region_type'] == 'SubtelomericRepeat') |
    (regions['region_type'] == 'SubtelomericHypervariable') |
    (regions['region_type'] == 'InternalHypervariable')
]

regions_non_accessible.head()

,region_chrom,region_start,region_stop,region_type
0,Pf3D7_01_v3,1,27336,SubtelomericRepeat
1,Pf3D7_01_v3,27337,92900,SubtelomericHypervariable
5,Pf3D7_01_v3,575901,616691,SubtelomericHypervariable
6,Pf3D7_01_v3,616692,640851,SubtelomericRepeat
7,Pf3D7_02_v3,1,23100,SubtelomericRepeat


In [12]:
chrom_variants = {}
assessibility_mask = []

# Initialize a dictionary to store chromosomal variants and accessibility mask
chrom_variants = {}
assessibility_mask = {}

for chrom in chrom_uniq:
    # Create a boolean mask for the current chromosome
    mask_chrom = variants_full['CHROM'] == chrom
    
    # Use compress to subset the variants for the current chromosome
    chrom_variants[chrom] = variants_full.compress(mask_chrom)
    
    # Subset regions that correspond to the current chromosome
    regions_non_accessible_chrom = regions_non_accessible[regions_non_accessible['region_chrom'] == chrom]
    
    # Extract the positions of the variants
    POS = chrom_variants[chrom]['POS'][:]
    
    # Create an empty mask (True means accessible, False means non-accessible)
    chrom_mask = np.ones(len(POS), dtype=bool)
    
    # Iterate through each region in the non-accessible regions for the current chromosome
    for _, region in regions_non_accessible_chrom.iterrows():
        region_start = region['region_start']
        region_stop = region['region_stop']
        
        # Check if any POS is within the current region range
        # Mark positions that lie between region_start and region_stop as False (non-accessible)
        mask = (POS >= region_start) & (POS <= region_stop)
        
        # Print positions that are being marked as non-accessible (optional)
        
        # Set the chrom_mask to False for all positions within the range
        chrom_mask[mask] = False
        
    # Store the mask for the current chromosome in the dictionary
    assessibility_mask[chrom] = chrom_mask

# Check if any positions are marked as non-accessible for any chromosome
for chrom, mask in assessibility_mask.items():
    if np.any(mask == False):
        print(f"Chromosome {chrom} has non-accessible positions.")
    else:
        print(f"Chromosome {chrom} has all positions accessible.")

Chromosome Pf3D7_01_v3 has all positions accessible.
Chromosome Pf3D7_02_v3 has all positions accessible.
Chromosome Pf3D7_03_v3 has all positions accessible.
Chromosome Pf3D7_04_v3 has all positions accessible.
Chromosome Pf3D7_05_v3 has all positions accessible.
Chromosome Pf3D7_06_v3 has all positions accessible.
Chromosome Pf3D7_07_v3 has all positions accessible.
Chromosome Pf3D7_08_v3 has all positions accessible.
Chromosome Pf3D7_09_v3 has all positions accessible.
Chromosome Pf3D7_10_v3 has all positions accessible.
Chromosome Pf3D7_11_v3 has all positions accessible.
Chromosome Pf3D7_12_v3 has all positions accessible.
Chromosome Pf3D7_13_v3 has all positions accessible.
Chromosome Pf3D7_14_v3 has all positions accessible.


In [13]:
chrom='Pf3D7_01_v3'
# Create a boolean mask for the current chromosome
mask_chrom = variants_full['CHROM'] == chrom
    
# Use compress to subset the variants for the current chromosome
chrom_variants[chrom] = variants_full.compress(mask_chrom)
regions_non_accessible_chrom = regions_non_accessible[regions_non_accessible['region_chrom'] == 'Pf3D7_01_v3']
print(regions_non_accessible_chrom)
POS= chrom_variants['Pf3D7_01_v3']['POS'][:]
print(POS) 
# Create an empty mask (True means accessible, False means non-accessible)
chrom_mask = np.ones(chrom_variants['Pf3D7_01_v3'].shape[0], dtype=bool)

len(chrom_mask)


  region_chrom  region_start  region_stop                region_type
0  Pf3D7_01_v3             1        27336         SubtelomericRepeat
1  Pf3D7_01_v3         27337        92900  SubtelomericHypervariable
5  Pf3D7_01_v3        575901       616691  SubtelomericHypervariable
6  Pf3D7_01_v3        616692       640851         SubtelomericRepeat
[ 92914  92918  92935 ... 575279 575284 575895]


9911

In [17]:
regions_non_accessible_chrom

,region_chrom,region_start,region_stop,region_type
0,Pf3D7_01_v3,1,27336,SubtelomericRepeat
1,Pf3D7_01_v3,27337,92900,SubtelomericHypervariable
5,Pf3D7_01_v3,575901,616691,SubtelomericHypervariable
6,Pf3D7_01_v3,616692,640851,SubtelomericRepeat


In [180]:
np.any(chrom_mask == False)

False

In [155]:
chrom_variants['Pf3D7_01_v3']['POS'][:]

array([ 92914,  92918,  92935, ..., 575279, 575284, 575895], dtype=int32)

In [14]:
# Calculate total length of each chromosome
chrom_lengths = chrom_regions.groupby('chrom')['end'].max()

# Convert chrom_lengths (Series) to DataFrame
chrom_lengths = pd.DataFrame(chrom_lengths).reset_index()

# Rename columns if necessary
chrom_lengths.columns = ['chrom', 'end']

In [15]:
# Initialize a dictionary to store chromosomal variants and accessibility mask
chrom_variants = {}
assessibility_mask = {}

for chrom in chrom_uniq:
    # Subset regions that correspond to the current chromosome
    regions_non_accessible_chrom = regions_non_accessible[regions_non_accessible['region_chrom'] == chrom]
    
    # Get the length of the chromosome (chrom_lengths is a DataFrame)
    chrom_length_for_mask = chrom_lengths[chrom_lengths['chrom'] == chrom]['end'].values[0]  # Get the value as integer
    
    # Create an empty mask (True means accessible, False means non-accessible)
    chrom_mask = np.ones(chrom_length_for_mask, dtype=bool)

    # Iterate through each region in the non-accessible regions for the current chromosome
    for _, region in regions_non_accessible_chrom.iterrows():

        region_start = region['region_start']
        region_stop = region['region_stop']
        
        # Adjust np.arange to start from 1 instead of 0
        mask = (np.arange(1, chrom_mask.size + 1) >= region_start) & (np.arange(1, chrom_mask.size + 1) <= region_stop)
    
        
        # Set the chrom_mask to False for all positions within the range
        chrom_mask[mask] = False
        
    # Store the mask for the current chromosome in the dictionary
    assessibility_mask[chrom] = chrom_mask

    

In [16]:
# Calculate total length of each chromosome
chrom_lengths = chrom_regions.groupby('chrom')['end'].max()

chrom_lengths_df = chrom_lengths.reset_index()
chrom_lengths_df.columns = ['chrom', 'end']

In [17]:
windows_store={}
# Iterate over all chromosomes
for chrom in chrom_uniq:
    # Get the accessibility mask for the current chromosome
    chrom_mask = assessibility_mask.get(chrom)
    
    # Get the length of the chromosome for defining 'stop'
    chrom_length_for_mask = chrom_lengths_df[chrom_lengths_df['chrom'] == chrom]['end'].values[0]
    
    # Define the start and stop positions
    start = 0
    stop = chrom_length_for_mask  # Stop position should be the length of the chromosome
    
    # Set window size and step size
    window_size = 1000 # Example window size in bases
    step_size = 500  # Step size in bases
    
    # Apply equally_accessible_windows using the accessibility mask for the current chromosome
    windows  = allel.equally_accessible_windows(
        chrom_mask, 
        size=window_size, 
        start=start, 
        stop=stop, 
        step=step_size
    )
    windows_store[chrom] = windows
    

In [17]:
assessibility_mask['Pf3D7_01_v3']
windows_store['Pf3D7_01_v3']

array([[ 92901,  93900],
       [ 93401,  94400],
       [ 93901,  94900],
       ...,
       [573901, 574900],
       [574401, 575400],
       [574901, 575900]], shape=(965, 2))

In [18]:
# Parameters
window_size = 1000
step_size = 500

sample_data_by_chrom = {}
windows_store = {}

total_vars = 0
total_vars_missing_cleaned = 0
total_vars_accessible = 0

for chrom in chrom_uniq:
    
    print(f"\nProcessing chromosome: {chrom}")
    
    # ----------------------------
    # 1. Subset by chromosome
    # ----------------------------
    mask_chrom = variants_full['CHROM'] == chrom
    
    chrom_variants = variants_full.compress(mask_chrom)
    chrom_pos = chrom_variants['POS'][:]
    chrom_gt = genotypes_full.compress(mask_chrom)
    
    total_vars += np.sum(mask_chrom)

    # ----------------------------
    # 2. Remove sites with missing genotypes
    # ----------------------------
    is_missing = chrom_gt.is_missing()[:]
    valid_sites_mask = ~np.any(is_missing, axis=1)
    
    chrom_gt = chrom_gt.compress(valid_sites_mask)
    chrom_variants = chrom_variants.compress(valid_sites_mask)
    chrom_pos = chrom_pos[valid_sites_mask]
    
    total_vars_missing_cleaned += np.sum(valid_sites_mask)
    
    print(f"Removed {np.sum(mask_chrom) - np.sum(valid_sites_mask)} missing sites")

    # ----------------------------
    # 3. Apply accessibility mask
    # ----------------------------
    chrom_mask = assessibility_mask.get(chrom)
    
    # Convert genomic positions to 0-based index for mask lookup
    pos_index = chrom_pos - 1
    
    # Keep only positions marked accessible (True)
    accessible_sites_mask = chrom_mask[pos_index]
    
    chrom_gt = chrom_gt.compress(accessible_sites_mask)
    chrom_variants = chrom_variants.compress(accessible_sites_mask)
    chrom_pos = chrom_pos[accessible_sites_mask]
    
    total_vars_accessible += np.sum(accessible_sites_mask)
    
    print(f"Retained {np.sum(accessible_sites_mask)} accessible sites")

    # ----------------------------
    # 4. Allele counts per subpopulation
    # ----------------------------
    ac_subpops = chrom_gt.count_alleles_subpops(subpops, max_allele=2)

    gt_valid_by_subpop = {}
    pos_by_subpop = {}

    target_subpops = [
        ('MaeSot_Thailand', 'GT_MaeSot', 'POS_MaeSot'),
        ('BinhPhuoc_Long_Vietnam', 'GT_Binh', 'POS_Binh')
    ]

    for subpop_key, gt_key, pos_key in target_subpops:
        
        # 1Extract subpopulation from filtered chrom_gt
        subpop_indices = subpops[subpop_key]
        gt_sub = chrom_gt.take(subpop_indices, axis=1)
        
        # Rank samples by missingness
        is_missing_sub = gt_sub.is_missing()[:]
        missing_per_sample = np.sum(is_missing_sub, axis=0)
        
        n50 = min(50, gt_sub.shape[1])
        top50_local_idx = np.argsort(missing_per_sample)[:n50]
        
        gt_top50 = gt_sub.take(top50_local_idx, axis=1)
        
        # Compute allele counts on top50 only
        ac_top50 = gt_top50.count_alleles(max_allele=2)
        is_seg = ac_top50.is_segregating()
        
        # Keep only segregating sites
        gt_seg = gt_top50.compress(is_seg, axis=0)
        pos_seg = chrom_pos[is_seg]
        
        gt_valid = pd.DataFrame(gt_seg).T
        
        gt_valid_by_subpop[gt_key] = gt_valid
        pos_by_subpop[pos_key] = pos_seg
        
        print(f"{gt_key}: {gt_valid.shape[0]} samples, {gt_valid.shape[1]} segregating sites")


    # ----------------------------
    # 5. Store genotype results
    # ----------------------------
    sample_data_by_chrom[chrom] = {
        'CHROM': chrom,
        'GT_MaeSot': gt_valid_by_subpop.get('GT_MaeSot', pd.DataFrame()),
        'POS_MaeSot': pos_by_subpop.get('POS_MaeSot', np.array([])),
        'GT_Binh': gt_valid_by_subpop.get('GT_Binh', pd.DataFrame()),
        'POS_Binh': pos_by_subpop.get('POS_Binh', np.array([]))
    }



Processing chromosome: Pf3D7_01_v3
Removed 2164 missing sites
Retained 7747 accessible sites
GT_MaeSot: 50 samples, 399 segregating sites
GT_Binh: 50 samples, 435 segregating sites

Processing chromosome: Pf3D7_02_v3
Removed 3882 missing sites
Retained 12387 accessible sites
GT_MaeSot: 50 samples, 366 segregating sites
GT_Binh: 50 samples, 420 segregating sites

Processing chromosome: Pf3D7_03_v3
Removed 4134 missing sites
Retained 16352 accessible sites
GT_MaeSot: 50 samples, 528 segregating sites
GT_Binh: 50 samples, 586 segregating sites

Processing chromosome: Pf3D7_04_v3
Removed 4753 missing sites
Retained 16716 accessible sites
GT_MaeSot: 50 samples, 795 segregating sites
GT_Binh: 50 samples, 891 segregating sites

Processing chromosome: Pf3D7_05_v3
Removed 6201 missing sites
Retained 21843 accessible sites
GT_MaeSot: 50 samples, 511 segregating sites
GT_Binh: 50 samples, 635 segregating sites

Processing chromosome: Pf3D7_06_v3
Removed 4886 missing sites
Retained 21352 accessib

In [21]:
output_dir= '/home/usriniva/Desktop/masters/Plasmodium_us_scripts/ld_hat_data/samples50_with_mask'

In [22]:
# === Write output with '?' for missing data ===
subpop = 'GT_MaeSot'
for chrom, data in sample_data_by_chrom.items():
    
    filename = f"{output_dir}/{subpop}_{chrom}_sample_gt_counts.txt"
    num_sequences = data[subpop].shape[0]
    num_sites = data[subpop].shape[1]
    phased_flag = 1  # Indicates phased genotypes
    line0 = f"{num_sequences} {num_sites} {phased_flag}"

    def split_line(line):
            return [line[i:i+2000] for i in range(0, len(line), 2000)]

    # Write header line (split if too long)
    with open(filename, "w") as f:
            for chunk in split_line(line0):
                f.write(chunk + "\n")

    # Write genotype data, replacing -1 (missing) with '?'
    with open(filename, "a") as f:
        for index, row in data[subpop].iterrows():
            line1 = f'>Genotype{index}'

            def replace_missing(x):
                try:
                    val = int(x)
                    return '?' if val == -1 else str(val)
                except Exception:
                    return '?'

            line2 = ''.join(replace_missing(x) for x in row.values)

            f.write(line1 + "\n")
            for chunk in split_line(line2):
                f.write(chunk + "\n")

In [23]:
# === Write output with '?' for missing data ===
subpop = 'GT_Binh'
for chrom, data in sample_data_by_chrom.items():
    
    filename = f"{output_dir}/{subpop}_{chrom}_sample_gt_counts.txt"
    num_sequences = data[subpop].shape[0]
    num_sites = data[subpop].shape[1]
    phased_flag = 1  # Indicates phased genotypes
    line0 = f"{num_sequences} {num_sites} {phased_flag}"

    def split_line(line):
            return [line[i:i+2000] for i in range(0, len(line), 2000)]

    # Write header line (split if too long)
    with open(filename, "w") as f:
            for chunk in split_line(line0):
                f.write(chunk + "\n")

    # Write genotype data, replacing -1 (missing) with '?'
    with open(filename, "a") as f:
        for index, row in data[subpop].iterrows():
            line1 = f'>Genotype{index}'

            def replace_missing(x):
                try:
                    val = int(x)
                    return '?' if val == -1 else str(val)
                except Exception:
                    return '?'

            line2 = ''.join(replace_missing(x) for x in row.values)

            f.write(line1 + "\n")
            for chunk in split_line(line2):
                f.write(chunk + "\n")


In [24]:
#subpop_key='MaeSot'
subpop_key='Binh'

In [25]:
for chrom, data in sample_data_by_chrom.items():
        #print(data)
        pos_key = f"POS_{subpop_key}"
        pos_array = data.get(pos_key, np.array([]))
        #print(pos_array)
        if len(pos_array) == 0:
            print(f"Skipping {chrom} - {subpop_key}: no segregating sites.")
            continue

        filename = f"{output_dir}/{chrom}_seg_sites_{subpop_key}.txt"

        # Number of SNPs and chromosomal length in kb
        num_snps = len(pos_array)
        chrom_range_bp = np.max(pos_array) / 1000  # in kb
        model_flag = "L"

        with open(filename, "w") as f:
            f.write(f"{num_snps} {chrom_range_bp} {model_flag}\n")
            for pos in pos_array:
                pos_kb = pos / 1000
                f.write(f"{pos_kb:.3f}\n")

        print(f"Saved: {filename}")

Saved: /home/usriniva/Desktop/masters/Plasmodium_us_scripts/ld_hat_data/samples50_with_mask/Pf3D7_01_v3_seg_sites_Binh.txt
Saved: /home/usriniva/Desktop/masters/Plasmodium_us_scripts/ld_hat_data/samples50_with_mask/Pf3D7_02_v3_seg_sites_Binh.txt
Saved: /home/usriniva/Desktop/masters/Plasmodium_us_scripts/ld_hat_data/samples50_with_mask/Pf3D7_03_v3_seg_sites_Binh.txt
Saved: /home/usriniva/Desktop/masters/Plasmodium_us_scripts/ld_hat_data/samples50_with_mask/Pf3D7_04_v3_seg_sites_Binh.txt
Saved: /home/usriniva/Desktop/masters/Plasmodium_us_scripts/ld_hat_data/samples50_with_mask/Pf3D7_05_v3_seg_sites_Binh.txt
Saved: /home/usriniva/Desktop/masters/Plasmodium_us_scripts/ld_hat_data/samples50_with_mask/Pf3D7_06_v3_seg_sites_Binh.txt
Saved: /home/usriniva/Desktop/masters/Plasmodium_us_scripts/ld_hat_data/samples50_with_mask/Pf3D7_07_v3_seg_sites_Binh.txt
Saved: /home/usriniva/Desktop/masters/Plasmodium_us_scripts/ld_hat_data/samples50_with_mask/Pf3D7_08_v3_seg_sites_Binh.txt
Saved: /home/usr